# 04 - Customer ID and Reference Date Check

**Project:** Online Retail II - Customer Churn Prediction
**Role:** Data Engineering & Cleaning

**Task 5:** Ensure the dataset has the correct customer IDs and reference dates.

This notebook checks only this task, in detail, with its own code:

| Section | What is checked |
|---|---|
| 1 | Load the data |
| 2 | Customer IDs: format, existence, uniqueness, no guests, same IDs in both datasets |
| 3 | Why customers are in or out at each reference date (every customer accounted for) |
| 4 | Reference dates: the planned dates, full windows, no partial month, correct sample labels |
| 5 | Each row's dates: features only from before the reference date, target only from the 3 months after |
| 6 | Example customer timelines |
| 7 | All checks and proof that they catch mistakes |
| 8 | Result file |

**Input:** `Working_Data/cleaned_transactions.csv.gz`, `Working_Data/Final_Modelling_Dataset.csv`, `Working_Data/Feature_Dataset.csv`
(created by notebooks 01 and 02)

**Output:** `Customer_ID_and_Reference_Date_Result.xlsx`

## 1. Load the data

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning)    # pandas notes about mixed int/float IDs in the mistake tests

WORK_DIR = 'Working_Data'
RESULT_FILE = 'Customer_ID_and_Reference_Date_Result.xlsx'
pd.set_option('display.max_colwidth', 120)

raw_ids = pd.read_csv('online_retail_II.csv', usecols=['Customer ID'])['Customer ID']
tx = pd.read_csv(f'{WORK_DIR}/cleaned_transactions.csv.gz', dtype={'Invoice': str, 'StockCode': str}, parse_dates=['InvoiceDate'])
modelling = pd.read_csv(f'{WORK_DIR}/Final_Modelling_Dataset.csv', parse_dates=['reference_date'])
features = pd.read_csv(f'{WORK_DIR}/Feature_Dataset.csv', parse_dates=['reference_date'])

print('Cleaned transactions   :', len(tx))
print('Final modelling dataset:', modelling.shape)
print('Feature dataset        :', features.shape)

Cleaned transactions   : 1003338
Final modelling dataset: (16495, 22)
Feature dataset        : (16495, 21)


In [2]:
# the planned design (from the project plan)
PLANNED_REFERENCE_DATES = pd.to_datetime(['2010-12-01', '2011-03-01', '2011-06-01', '2011-09-01'])
OUT_OF_TIME_DATE = PLANNED_REFERENCE_DATES.max()
OBS_MONTHS, TARGET_MONTHS, MIN_HISTORY_DAYS = 12, 3, 30

# orders of known customers (an order's time = the first time on its invoice)
known = tx.dropna(subset=['CustomerID'])
orders = known.groupby('Invoice').agg(customer_id=('CustomerID', 'first'), order_time=('InvoiceDate', 'min')).reset_index()
orders['customer_id'] = orders['customer_id'].astype(int)
orders['order_day'] = orders['order_time'].dt.normalize()
print('Orders of known customers:', len(orders), '| customers:', orders['customer_id'].nunique())

Orders of known customers: 36592 | customers: 5852


## 2. Customer IDs

The raw file stores Customer ID as a decimal number (for example `13085.0`). The ID must become a whole number with no change in value.

In [3]:
id_checks = {}
raw_known = raw_ids.dropna()
id_checks['Raw IDs are whole numbers (13085.0 -> 13085 loses nothing)'] = (
    (raw_known % 1 == 0).all(), f'{(raw_known % 1 != 0).sum()} raw IDs with decimals')

for name, d in [('modelling dataset', modelling), ('feature dataset', features)]:
    ids = d['customer_id']
    id_checks[f'No empty customer_id in the {name}'] = (ids.notnull().all(), f'{ids.isnull().sum()} empty')
    id_checks[f'customer_id is a whole number in the {name}'] = (
        pd.api.types.is_integer_dtype(ids) or (ids.dropna() % 1 == 0).all(), str(ids.dtype))
    id_checks[f'customer_id has exactly 5 digits in the {name}'] = (
        ids.dropna().between(10000, 99999).all(), f'{ids.min()} to {ids.max()}')

tx_customers = set(orders['customer_id'])
unknown_ids = set(modelling['customer_id'].dropna().astype(int)) - tx_customers
id_checks['Every customer_id exists in the cleaned transactions'] = (len(unknown_ids) == 0, f'{len(unknown_ids)} unknown IDs')
id_checks['No guest rows (all IDs come from known customers)'] = (
    modelling['customer_id'].notnull().all() and len(unknown_ids) == 0, f"{tx['CustomerID'].isnull().sum():,} guest rows excluded")
dup = modelling.duplicated(['customer_id', 'reference_date']).sum()
id_checks['Each customer appears at most once per reference date'] = (dup == 0, f'{dup} duplicates')
id_checks['Feature dataset has the same customer IDs in the same order'] = (
    features['customer_id'].equals(modelling['customer_id']), '')
id_checks['Feature dataset has the same (customer, reference date) pairs'] = (
    features[['customer_id', 'reference_date']].equals(modelling[['customer_id', 'reference_date']]), '')

pd.DataFrame([{'Check': k, 'Result': 'PASS' if v[0] else 'FAIL', 'Detail': v[1]} for k, v in id_checks.items()])

,Check,Result,Detail
0,Raw IDs are whole numbers (13085.0 -> 13085 loses nothing),PASS,0 raw IDs with decimals
1,No empty customer_id in the modelling dataset,PASS,0 empty
2,customer_id is a whole number in the modelling dataset,PASS,int64
3,customer_id has exactly 5 digits in the modelling dataset,PASS,12346 to 18287
4,No empty customer_id in the feature dataset,PASS,0 empty
5,customer_id is a whole number in the feature dataset,PASS,int64
6,customer_id has exactly 5 digits in the feature dataset,PASS,12346 to 18287
7,Every customer_id exists in the cleaned transactions,PASS,0 unknown IDs
8,No guest rows (all IDs come from known customers),PASS,"226,761 guest rows excluded"
9,Each customer appears at most once per reference date,PASS,0 duplicates


In [4]:
appear = modelling.groupby('customer_id')['reference_date'].nunique().value_counts().sort_index()
appearances = pd.DataFrame({'reference dates the customer appears at': appear.index, 'customers': appear.values})
print('Different customers in the dataset:', modelling['customer_id'].nunique(), 'out of', len(tx_customers))
appearances

Different customers in the dataset: 5129 out of 5852


,reference dates the customer appears at,customers
0,1,557
1,2,750
2,3,850
3,4,2972


A customer can appear at several reference dates (once at each). That is expected: each row describes the customer on a different date.

## 3. Every customer accounted for at every reference date

For each reference date, every one of the known customers must be either **in** the dataset or **out** for one clear reason:

| Reason | Rule |
|---|---|
| Not a customer yet | First order on or after the reference date |
| Already lapsed | No order in the 12 months before the reference date |
| Too new | First order less than 30 days before the reference date |
| **Eligible** | Everything else: must be in the dataset |

In [5]:
first_day = orders.groupby('customer_id')['order_day'].min()
status_rows, eligible_sets = [], {}
for ref in PLANNED_REFERENCE_DATES:
    obs_start = ref - pd.DateOffset(months=OBS_MONTHS)
    in_obs = set(orders[(orders['order_time'] >= obs_start) & (orders['order_time'] < ref)]['customer_id'])
    status = pd.Series('Eligible', index=first_day.index)
    status[first_day >= ref] = 'Not a customer yet'
    status[(first_day < ref) & ~status.index.isin(in_obs)] = 'Already lapsed (no order in 12 months)'
    status[(status == 'Eligible') & ((ref - first_day).dt.days < MIN_HISTORY_DAYS)] = 'Too new (< 30 days history)'
    eligible_sets[ref] = set(status[status == 'Eligible'].index)
    in_dataset = set(modelling.loc[modelling['reference_date'] == ref, 'customer_id'])
    row = {'reference_date': ref.date(), **status.value_counts().to_dict(), 'total customers': len(status),
           'rows in dataset': len(in_dataset),
           'eligible but missing from dataset': len(eligible_sets[ref] - in_dataset),
           'in dataset but not eligible': len(in_dataset - eligible_sets[ref])}
    status_rows.append(row)

customer_status = pd.DataFrame(status_rows).fillna(0)
cols = ['reference_date', 'Not a customer yet', 'Already lapsed (no order in 12 months)', 'Too new (< 30 days history)',
        'Eligible', 'total customers', 'rows in dataset', 'eligible but missing from dataset', 'in dataset but not eligible']
customer_status = customer_status[cols]
customer_status[cols[1:]] = customer_status[cols[1:]].astype(int)
customer_status

,reference_date,Not a customer yet,Already lapsed (no order in 12 months),Too new (< 30 days history),Eligible,total customers,rows in dataset,eligible but missing from dataset,in dataset but not eligible
0,2010-12-01,1613,0,318,3921,5852,3921,0,0
1,2011-03-01,1340,252,128,4132,5852,4132,0,0
2,2011-06-01,944,584,110,4214,5852,4214,0,0
3,2011-09-01,628,901,95,4228,5852,4228,0,0


In [6]:
account_checks = {
    'Every customer has exactly one status at every reference date':
        ((customer_status[cols[1:5]].sum(axis=1) == customer_status['total customers']).all(),
         f"{customer_status['total customers'].iloc[0]:,} customers at each date"),
    'Rows in dataset = eligible customers at every reference date':
        ((customer_status['rows in dataset'] == customer_status['Eligible']).all(),
         ', '.join(map(str, customer_status['rows in dataset']))),
    'No eligible customer is missing': ((customer_status['eligible but missing from dataset'] == 0).all(), ''),
    'No ineligible customer is included': ((customer_status['in dataset but not eligible'] == 0).all(), ''),
}
pd.DataFrame([{'Check': k, 'Result': 'PASS' if v[0] else 'FAIL', 'Detail': v[1]} for k, v in account_checks.items()])

,Check,Result,Detail
0,Every customer has exactly one status at every reference date,PASS,"5,852 customers at each date"
1,Rows in dataset = eligible customers at every reference date,PASS,"3921, 4132, 4214, 4228"
2,No eligible customer is missing,PASS,
3,No ineligible customer is included,PASS,


## 4. Reference dates

Each reference date must be one of the 4 planned dates, the 1st of a month, 3 months apart, and its windows must sit fully inside
the data without touching the partial month (December 2011, data ends 9 December).

In [7]:
data_start = tx['InvoiceDate'].min().normalize()
data_end = tx['InvoiceDate'].max().normalize()
partial_month_start = data_end.to_period('M').to_timestamp()   # December 2011 is not complete
trading_days = set(tx['InvoiceDate'].dt.normalize())

windows = []
for ref in PLANNED_REFERENCE_DATES:
    obs_start = ref - pd.DateOffset(months=OBS_MONTHS)
    target_end = ref + pd.DateOffset(months=TARGET_MONTHS)
    windows.append({
        'reference_date': ref.date(),
        'sample': 'out_of_time_test' if ref == OUT_OF_TIME_DATE else 'development',
        'observation_window': f"{obs_start.date()} to {(ref - pd.Timedelta(days=1)).date()}",
        'target_window': f"{ref.date()} to {(target_end - pd.Timedelta(days=1)).date()}",
        'trading_days_in_observation': sum(obs_start <= d < ref for d in trading_days),
        'trading_days_in_target': sum(ref <= d < target_end for d in trading_days),
        'orders_in_target_window': int(((orders['order_time'] >= ref) & (orders['order_time'] < target_end)).sum()),
        'observation_starts_after_data_start': obs_start >= data_start,
        'target_ends_before_partial_month': target_end <= partial_month_start,
    })
reference_windows = pd.DataFrame(windows)
print('Data:', data_start.date(), 'to', data_end.date(), '| partial month starts', partial_month_start.date())
reference_windows

Data: 2009-12-01 to 2011-12-09 | partial month starts 2011-12-01


,reference_date,sample,observation_window,target_window,trading_days_in_observation,trading_days_in_target,orders_in_target_window,observation_starts_after_data_start,target_ends_before_partial_month
0,2010-12-01,development,2009-12-01 to 2010-11-30,2010-12-01 to 2011-02-28,299,68,3368,True,True
1,2011-03-01,development,2010-03-01 to 2011-02-28,2011-03-01 to 2011-05-31,298,73,3995,True,True
2,2011-06-01,development,2010-06-01 to 2011-05-31,2011-06-01 to 2011-08-31,297,78,3978,True,True
3,2011-09-01,out_of_time_test,2010-09-01 to 2011-08-31,2011-09-01 to 2011-11-30,297,78,6284,True,True


In [8]:
refs_in_data = sorted(modelling['reference_date'].unique())
expected_sample = np.where(modelling['reference_date'] == OUT_OF_TIME_DATE, 'out_of_time_test', 'development')
ref_checks = {
    'Reference dates are exactly the 4 planned dates': (
        set(modelling['reference_date']) == set(PLANNED_REFERENCE_DATES), ', '.join(str(pd.Timestamp(d).date()) for d in refs_in_data)),
    'No empty reference_date': (modelling['reference_date'].notnull().all(), ''),
    'Every reference date is the 1st of a month': (all(pd.Timestamp(d).day == 1 for d in refs_in_data), ''),
    'Reference dates are 3 months apart': (
        all(PLANNED_REFERENCE_DATES[i] + pd.DateOffset(months=3) == PLANNED_REFERENCE_DATES[i + 1] for i in range(3)), ''),
    'Every observation window starts after the first data date': (reference_windows['observation_starts_after_data_start'].all(), ''),
    'Every target window ends before the partial month': (reference_windows['target_ends_before_partial_month'].all(), ''),
    'Every window has sales on its trading days (no empty window)': (
        (reference_windows['trading_days_in_observation'] > 200).all() and (reference_windows['trading_days_in_target'] > 60).all(),
        f"min {reference_windows['trading_days_in_observation'].min()} / {reference_windows['trading_days_in_target'].min()} days"),
    'Development target windows end on or before the out-of-time date': (
        max(d + pd.DateOffset(months=TARGET_MONTHS) for d in PLANNED_REFERENCE_DATES[:-1]) <= OUT_OF_TIME_DATE, ''),
    'sample label matches the reference date on every row': ((modelling['sample'] == expected_sample).all(),
                                                            f"{(modelling['sample'] != expected_sample).sum()} wrong"),
    'Feature dataset has the same reference dates and sample labels': (
        features[['reference_date', 'sample']].equals(modelling[['reference_date', 'sample']]), ''),
}
pd.DataFrame([{'Check': k, 'Result': 'PASS' if v[0] else 'FAIL', 'Detail': v[1]} for k, v in ref_checks.items()])

,Check,Result,Detail
0,Reference dates are exactly the 4 planned dates,PASS,"2010-12-01, 2011-03-01, 2011-06-01, 2011-09-01"
1,No empty reference_date,PASS,
2,Every reference date is the 1st of a month,PASS,
3,Reference dates are 3 months apart,PASS,
4,Every observation window starts after the first data date,PASS,
5,Every target window ends before the partial month,PASS,
6,Every window has sales on its trading days (no empty window),PASS,min 297 / 68 days
7,Development target windows end on or before the out-of-time date,PASS,
8,sample label matches the reference date on every row,PASS,0 wrong
9,Feature dataset has the same reference dates and sample labels,PASS,


## 5. Each row's dates are correct

For every row, using the customer's real order history:
- the **last order before the reference date** must give exactly the row's `recency_days`
- the **first order** must be at least 30 days before the reference date
- `churn_next_3m` must be 0 if the customer ordered in the 3 months after the reference date, and 1 if not

In [9]:
row_checks_parts = []
for ref in PLANNED_REFERENCE_DATES:
    rows = modelling[modelling['reference_date'] == ref][['customer_id', 'recency_days', 'churn_next_3m']]
    before = orders[orders['order_time'] < ref]
    last_before = before.groupby('customer_id')['order_day'].max()
    first_before = before.groupby('customer_id')['order_day'].min()
    target = orders[(orders['order_time'] >= ref) & (orders['order_time'] < ref + pd.DateOffset(months=TARGET_MONTHS))]
    r = rows.set_index('customer_id')
    r['last_order_before_ref'] = last_before.reindex(r.index)
    r['first_order'] = first_before.reindex(r.index)
    r['recency_from_history'] = (ref - r['last_order_before_ref']).dt.days
    r['days_of_history'] = (ref - r['first_order']).dt.days
    r['ordered_in_target_window'] = r.index.isin(target['customer_id'])
    r['reference_date'] = ref
    row_checks_parts.append(r.reset_index())
row_dates = pd.concat(row_checks_parts, ignore_index=True)

date_checks = {
    'Every customer has an order before their reference date': (row_dates['last_order_before_ref'].notnull().all(), ''),
    'recency_days = days from the last order to the reference date (every row)': (
        (row_dates['recency_days'] == row_dates['recency_from_history']).all(),
        f"{(row_dates['recency_days'] != row_dates['recency_from_history']).sum()} rows differ"),
    'Last order used is before the reference date (recency >= 1)': ((row_dates['recency_from_history'] >= 1).all(), ''),
    'Last order is inside the 12-month observation window (recency <= 365)': ((row_dates['recency_from_history'] <= 365).all(), ''),
    'First order is at least 30 days before the reference date': (
        (row_dates['days_of_history'] >= MIN_HISTORY_DAYS).all(), f"min {row_dates['days_of_history'].min()} days"),
    'churn = 0 exactly when the customer ordered in the target window': (
        ((row_dates['churn_next_3m'] == 0) == row_dates['ordered_in_target_window']).all(),
        f"{((row_dates['churn_next_3m'] == 0) != row_dates['ordered_in_target_window']).sum()} rows differ"),
}
pd.DataFrame([{'Check': k, 'Result': 'PASS' if v[0] else 'FAIL', 'Detail': v[1]} for k, v in date_checks.items()])

,Check,Result,Detail
0,Every customer has an order before their reference date,PASS,
1,recency_days = days from the last order to the reference date (every row),PASS,0 rows differ
2,Last order used is before the reference date (recency >= 1),PASS,
3,Last order is inside the 12-month observation window (recency <= 365),PASS,
4,First order is at least 30 days before the reference date,PASS,min 30 days
5,churn = 0 exactly when the customer ordered in the target window,PASS,0 rows differ


## 6. Example customer timelines

Three real customers, to show how one customer gets one row per reference date and how the dates decide each value.

In [10]:
counts = modelling.groupby('customer_id')['reference_date'].nunique()
churn_mix = modelling.groupby('customer_id')['churn_next_3m'].nunique()
example_ids = [counts[(counts == 4) & (churn_mix == 2)].index[0],      # at all 4 dates, churn changes
               counts[counts == 2].index[0],                            # at 2 dates only
               counts[counts == 1].index[0]]                            # at 1 date only
examples = []
for cid in example_ids:
    hist = orders[orders['customer_id'] == cid].sort_values('order_time')
    for ref in PLANNED_REFERENCE_DATES:
        row = modelling[(modelling['customer_id'] == cid) & (modelling['reference_date'] == ref)]
        before = hist[hist['order_time'] < ref]
        after = hist[(hist['order_time'] >= ref) & (hist['order_time'] < ref + pd.DateOffset(months=3))]
        examples.append({
            'customer_id': cid, 'reference_date': ref.date(),
            'in_dataset': 'yes' if len(row) else 'no',
            'first_order': before['order_day'].min().date() if len(before) else 'none yet',
            'last_order_before_ref': before['order_day'].max().date() if len(before) else 'none yet',
            'orders_in_next_3_months': len(after),
            'recency_days': int(row['recency_days'].iloc[0]) if len(row) else '',
            'churn_next_3m': int(row['churn_next_3m'].iloc[0]) if len(row) else '',
        })
example_timelines = pd.DataFrame(examples)
example_timelines

,customer_id,reference_date,in_dataset,first_order,last_order_before_ref,orders_in_next_3_months,recency_days,churn_next_3m
0,12348,2010-12-01,yes,2010-09-27,2010-09-27,2,65,0
1,12348,2011-03-01,yes,2010-09-27,2011-01-25,1,35,0
2,12348,2011-06-01,yes,2010-09-27,2011-04-05,0,57,1
3,12348,2011-09-01,yes,2010-09-27,2011-04-05,1,149,0
4,12350,2010-12-01,no,none yet,none yet,1,,
5,12350,2011-03-01,no,2011-02-02,2011-02-02,0,,
6,12350,2011-06-01,yes,2011-02-02,2011-02-02,0,119,1
7,12350,2011-09-01,yes,2011-02-02,2011-02-02,0,211,1
8,12394,2010-12-01,no,none yet,none yet,0,,
9,12394,2011-03-01,no,none yet,none yet,1,,


How to read it: when a customer is in the dataset, `recency_days` is the gap between `last_order_before_ref` and the reference date,
and `churn_next_3m` is 1 only when `orders_in_next_3_months` is 0. A customer is left out when they had not ordered yet,
had no order for 12 months, or had less than 30 days of history.

## 7. All checks, and proof that they catch mistakes

All the checks above are run again in one function. Then a mistake is put into a copy of the dataset, and the same checks must FAIL.

In [11]:
def check_ids_and_dates(md_, fd_):
    out = {}
    ids = md_['customer_id']
    out['customer_id is never empty'] = ids.notnull().all()
    out['customer_id is a whole 5-digit number'] = (ids.dropna() % 1 == 0).all() and ids.dropna().between(10000, 99999).all()
    out['Every customer_id exists in the transactions'] = set(ids.dropna().astype(int)) <= tx_customers
    out['One row per customer per reference date'] = not md_.duplicated(['customer_id', 'reference_date']).any()
    out['Feature dataset has the same IDs and dates'] = (
        len(fd_) == len(md_) and fd_[['customer_id', 'reference_date']].reset_index(drop=True)
        .equals(md_[['customer_id', 'reference_date']].reset_index(drop=True)))
    out['Reference dates are the 4 planned dates'] = set(md_['reference_date'].dropna()) == set(PLANNED_REFERENCE_DATES)
    out['sample label matches the reference date'] = (
        md_['sample'] == np.where(md_['reference_date'] == OUT_OF_TIME_DATE, 'out_of_time_test', 'development')).all()
    ok_rows = True
    for ref in PLANNED_REFERENCE_DATES:
        rows = md_[md_['reference_date'] == ref]
        if set(rows['customer_id']) != eligible_sets[ref] or len(rows) != len(eligible_sets[ref]):
            ok_rows = False
    out['Customers at each reference date = eligible customers'] = ok_rows
    m = md_.merge(row_dates[['customer_id', 'reference_date', 'recency_from_history', 'ordered_in_target_window']],
                  on=['customer_id', 'reference_date'], how='left')
    out['recency_days matches the order history'] = (m['recency_days'] == m['recency_from_history']).all()
    out['churn matches the target window'] = ((m['churn_next_3m'] == 0) == m['ordered_in_target_window'].fillna(False)).all()
    return out

all_checks = {**id_checks, **account_checks, **ref_checks, **date_checks}
final_checks = check_ids_and_dates(modelling, features)
print('Detailed checks passed:', sum(v[0] for v in all_checks.values()), 'out of', len(all_checks))
print('Summary function checks passed:', sum(final_checks.values()), 'out of', len(final_checks))
assert all(v[0] for v in all_checks.values()) and all(final_checks.values())

Detailed checks passed: 32 out of 32
Summary function checks passed: 10 out of 10


In [12]:
mistakes = []
def try_mistake(name, md_=None, fd_=None):
    res = check_ids_and_dates(modelling if md_ is None else md_, features if fd_ is None else fd_)
    failed = [k for k, v in res.items() if not v]
    mistakes.append({'Mistake added': name, 'Checks failed': len(failed), 'Caught': len(failed) > 0, 'Failed checks': '; '.join(failed)})

def changed(col, value, row=5):
    m = modelling.copy()
    if col == 'customer_id':
        m[col] = m[col].astype(float)        # so an empty or decimal ID can be put in
    m.loc[row, col] = value
    return m

try_mistake('Empty customer_id', changed('customer_id', np.nan))
try_mistake('Customer ID with a decimal (13085.5)', changed('customer_id', 13085.5))
try_mistake('4-digit customer ID (1234)', changed('customer_id', 1234))
try_mistake('Customer ID not in the transactions (99999)', changed('customer_id', 99999))
try_mistake('Same customer twice at one reference date', pd.concat([modelling, modelling.iloc[[0]]], ignore_index=True))
try_mistake('Wrong reference date (15-Sep-2011)', changed('reference_date', pd.Timestamp('2011-09-15')))
try_mistake('Reference date in the partial month (01-Dec-2011)', changed('reference_date', pd.Timestamp('2011-12-01')))
try_mistake('Wrong sample label', changed('sample', 'out_of_time_test', row=0))
try_mistake('IDs out of sync between the two datasets',
            fd_=features.assign(customer_id=features['customer_id'].shift(1).fillna(features['customer_id'].iloc[0])))
try_mistake('Recency calculated with the wrong reference date', modelling.assign(recency_days=modelling['recency_days'] + 30))
mistake_tests = pd.DataFrame(mistakes)
print('Mistakes caught:', mistake_tests['Caught'].sum(), 'out of', len(mistake_tests))
assert mistake_tests['Caught'].all()
mistake_tests

Mistakes caught: 10 out of 10


,Mistake added,Checks failed,Caught,Failed checks
0,Empty customer_id,4,True,customer_id is never empty; Feature dataset has the same IDs and dates; Customers at each reference date = eligible ...
1,Customer ID with a decimal (13085.5),4,True,customer_id is a whole 5-digit number; Feature dataset has the same IDs and dates; Customers at each reference date ...
2,4-digit customer ID (1234),5,True,customer_id is a whole 5-digit number; Every customer_id exists in the transactions; Feature dataset has the same ID...
3,Customer ID not in the transactions (99999),4,True,Every customer_id exists in the transactions; Feature dataset has the same IDs and dates; Customers at each referenc...
4,Same customer twice at one reference date,3,True,One row per customer per reference date; Feature dataset has the same IDs and dates; Customers at each reference dat...
5,Wrong reference date (15-Sep-2011),4,True,Feature dataset has the same IDs and dates; Reference dates are the 4 planned dates; Customers at each reference dat...
6,Reference date in the partial month (01-Dec-2011),4,True,Feature dataset has the same IDs and dates; Reference dates are the 4 planned dates; Customers at each reference dat...
7,Wrong sample label,1,True,sample label matches the reference date
8,IDs out of sync between the two datasets,1,True,Feature dataset has the same IDs and dates
9,Recency calculated with the wrong reference date,1,True,recency_days matches the order history


## 8. Result file

In [13]:
check_table = pd.concat([
    pd.DataFrame([{'Section': sec, 'Check': k, 'Result': 'PASS' if v[0] else 'FAIL', 'Detail': v[1]} for k, v in checks.items()])
    for sec, checks in [('2. Customer IDs', id_checks), ('3. Every customer accounted for', account_checks),
                        ('4. Reference dates', ref_checks), ('5. Row dates', date_checks)]], ignore_index=True)
summary = check_table.groupby('Section', sort=False)['Result'].agg(checks='count', passed=lambda s: (s == 'PASS').sum()).reset_index()
summary = pd.concat([summary, pd.DataFrame([
    {'Section': '7. Mistakes caught', 'checks': len(mistake_tests), 'passed': int(mistake_tests['Caught'].sum())},
    {'Section': 'TOTAL', 'checks': len(check_table) + len(mistake_tests),
     'passed': int((check_table['Result'] == 'PASS').sum() + mistake_tests['Caught'].sum())}])], ignore_index=True)
summary['result'] = np.where(summary['checks'] == summary['passed'], 'PASS', 'FAIL')

sheets = {'Summary': summary, 'All_Checks': check_table, 'Customer_Status_by_Date': customer_status,
          'Reference_Windows': reference_windows, 'Customer_Appearances': appearances,
          'Example_Timelines': example_timelines, 'Mistake_Tests': mistake_tests}
with pd.ExcelWriter(RESULT_FILE) as writer:
    for name, table in sheets.items():
        table.to_excel(writer, sheet_name=name, index=False)
saved = pd.read_excel(RESULT_FILE, sheet_name=None)
for name, table in sheets.items():
    assert saved[name].shape == table.shape, f'sheet {name} is wrong'
print('Saved', RESULT_FILE, 'with', len(saved), 'sheets:', ', '.join(saved))
summary

Saved Customer_ID_and_Reference_Date_Result.xlsx with 7 sheets: Summary, All_Checks, Customer_Status_by_Date, Reference_Windows, Customer_Appearances, Example_Timelines, Mistake_Tests


,Section,checks,passed,result
0,2. Customer IDs,12,12,PASS
1,3. Every customer accounted for,4,4,PASS
2,4. Reference dates,10,10,PASS
3,5. Row dates,6,6,PASS
4,7. Mistakes caught,10,10,PASS
5,TOTAL,42,42,PASS


## Summary

| Item | Result |
|---|---|
| Customer IDs | Whole 5-digit numbers (12346 to 18287); raw IDs like 13085.0 convert with no change; 0 empty, 0 unknown, 0 guests |
| Customers in the dataset | 5,129 of the 5,852 known customers; 2,972 appear at all 4 reference dates (one row per date, 0 duplicates) |
| Every customer accounted for | At every reference date all 5,852 customers are either eligible (in the dataset) or out for one reason; 0 missing, 0 wrongly included |
| Example: 01-Sep-2011 | 628 not a customer yet, 901 already lapsed, 95 too new, **4,228 eligible = 4,228 rows** |
| Reference dates | Exactly 01-Dec-2010, 01-Mar-2011, 01-Jun-2011 (development) and 01-Sep-2011 (out-of-time test), 3 months apart |
| Windows | 297-299 trading days in each observation window, 68-78 in each target window; none touches the partial December 2011 |
| Row dates | recency_days matches the real order history on every row; churn matches the 3-month window on every row |
| Checks | 32 out of 32 passed; 10 out of 10 deliberate mistakes caught (**42 / 42 PASS**) |

**Conclusion:** the final modelling dataset and the feature dataset have the correct customer IDs and the correct reference dates.

**Result file:** `Customer_ID_and_Reference_Date_Result.xlsx` (7 sheets)

In [14]:
sep = customer_status.set_index('reference_date').loc[pd.Timestamp('2011-09-01').date()]
facts = [('Lowest ID', 12346, int(modelling['customer_id'].min())), ('Highest ID', 18287, int(modelling['customer_id'].max())),
         ('Customers in dataset', 5129, modelling['customer_id'].nunique()), ('Known customers', 5852, len(tx_customers)),
         ('At all 4 dates', 2972, int(appearances.loc[appearances.iloc[:, 0] == 4, 'customers'].iloc[0])),
         ('Sep-2011 not yet', 628, int(sep['Not a customer yet'])), ('Sep-2011 lapsed', 901, int(sep['Already lapsed (no order in 12 months)'])),
         ('Sep-2011 too new', 95, int(sep['Too new (< 30 days history)'])), ('Sep-2011 eligible', 4228, int(sep['Eligible'])),
         ('Min observation trading days', 297, int(reference_windows['trading_days_in_observation'].min())),
         ('Max observation trading days', 299, int(reference_windows['trading_days_in_observation'].max())),
         ('Min target trading days', 68, int(reference_windows['trading_days_in_target'].min())),
         ('Max target trading days', 78, int(reference_windows['trading_days_in_target'].max())),
         ('Checks passed', 32, int((check_table['Result'] == 'PASS').sum())), ('Mistakes caught', 10, int(mistake_tests['Caught'].sum())),
         ('Result sheets', 7, len(saved))]
facts = pd.DataFrame(facts, columns=['Item', 'Reported', 'Notebook output'])
facts['Match'] = facts['Reported'].astype(str) == facts['Notebook output'].astype(str)
print('Summary numbers matching the outputs:', facts['Match'].sum(), 'out of', len(facts))
assert facts['Match'].all()
facts

Summary numbers matching the outputs: 16 out of 16


,Item,Reported,Notebook output,Match
0,Lowest ID,12346,12346,True
1,Highest ID,18287,18287,True
2,Customers in dataset,5129,5129,True
3,Known customers,5852,5852,True
4,At all 4 dates,2972,2972,True
5,Sep-2011 not yet,628,628,True
6,Sep-2011 lapsed,901,901,True
7,Sep-2011 too new,95,95,True
8,Sep-2011 eligible,4228,4228,True
9,Min observation trading days,297,297,True
